In [ ]:
import polars as pl
import geopandas as gpd

import sys
from pathlib import Path
import polars as pl 
sys.path.append(str(Path().resolve().parent))

from scripts.config import CSV_PATH, DATA_PATH, SAVE_FIGURE, FIGURE_DIR, TIF_PATH


In [ ]:
# from scripts.config import CSV_PATH, DATA_PATH, SAVE_FIGURE, FIGURE_DIR, TIF_PATH
# from scripts.tdlinx.data_loading import load_data, load_geodata


In [ ]:
# gdf, kreis_gdf, bundesland_gdf = load_geodata()

In [ ]:
# import geopandas as gpd
# from pathlib import Path
# import pandas as pd

# # 2. Get the bounding box of Germany to use as a spatial pre-filter
# bbox = tuple(gdf.total_bounds)  # (minx, miny, maxx, maxy)

# # 3. Load each grid file filtered to Germany's bounding box
# grid_files = list(Path(POP_SHAPEFILE_PATH).glob("*.shp"))


In [ ]:
# grid_files

In [ ]:

# grids = []
# sample = gpd.read_file(grid_files[0], rows=1)
# grid_crs = sample.crs

# germany_in_grid_crs = gdf.to_crs(grid_crs)
# bbox = tuple(germany_in_grid_crs.total_bounds)  # now in Mollweide coordinates

# for f in grid_files:
#     chunk = gpd.read_file(f, bbox=bbox)  # spatial pre-filter via bbox
#     print(chunk.head())
#     grids.append(chunk)


In [ ]:
# # 4. Combine all grid tiles
# grid = pd.concat(grids, ignore_index=True)

# # 5. Clip precisely to Germany's borders (removes bbox corners outside Germany)
# # grid = gpd.clip(grid, gdf)

In [ ]:
# grids[1].to_crs("EPSG:31467")

In [ ]:
import rasterio

SOURCE_FILE = r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project\\data"
files = [
    SOURCE_FILE + r"\ghsl raster files\GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R3_C19.tif",
    SOURCE_FILE + r"\ghsl raster files\GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R3_C20.tif",
    SOURCE_FILE + r"\ghsl raster files\GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R4_C19.tif",
    SOURCE_FILE + r"\ghsl raster files\GHS_POP_E2025_GLOBE_R2023A_54009_100_V1_0_R4_C20.tif"
]

for f in files:
    with rasterio.open(f) as src:
        print(f"{f.split('/')[-1]}")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
        print(f"  Shape: {src.shape}")
        print(f"  Nodata: {src.nodata}")
        print(f"  Dtype: {src.dtypes[0]}")
        print(f"  Scales: {src.scales}")


In [ ]:
import rasterio
import numpy as np

def inspect_raster(path: str, sample_points: dict = None):
    """
    Print a full summary of a raster's metadata and sample its values.
    
    sample_points: dict of {label: (lon, lat)} in WGS84 to sample at,
                   or None to skip sampling.
    """
    with rasterio.open(path) as src:
        print(f"=== {path.split('/')[-1]} ===\n")
        
        # Basic metadata
        print(f"CRS:          {src.crs}")
        print(f"Shape:        {src.shape}  (rows × cols)")
        print(f"Resolution:   {src.res}  (x, y in CRS units)")
        print(f"Bounds:       {src.bounds}")
        print(f"Dtype:        {src.dtypes[0]}")
        print(f"Nodata value: {src.nodata}")
        print(f"Bands:        {src.count}")
        print(f"Transform:")
        print(f"  {src.transform}")

        # Extent in WGS84 for reference
        from rasterio.warp import transform_bounds
        wgs = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
        print(f"\nBounds in WGS84 (lon_min, lat_min, lon_max, lat_max):")
        print(f"  {wgs}")

        # Value distribution — read a downsampled overview
        # Reading full raster into memory may be too much; use decimation
        n_rows, n_cols = src.shape
        # stride = max(1, n_rows // 500)  # sample ~500 rows worth
        stride = n_rows
        # In the diagnostic function, change:
        arr = src.read(
            1,
            # out_shape=(n_rows // stride, n_cols // stride),
            out_shape=(n_rows , n_cols),
            resampling=rasterio.enums.Resampling.nearest,
        )
        valid = arr[(arr != src.nodata) & np.isfinite(arr)]

        print(f"\nValue statistics (from decimated read):")
        print(f"  Non-nodata cells: {valid.size:,}")
        if valid.size > 0:
            print(f"  Min:      {valid.min():.3f}")
            print(f"  Max:      {valid.max():.3f}")
            print(f"  Mean:     {valid.mean():.3f}")
            print(f"  Median:   {np.median(valid):.3f}")
            print(f"  Sum:      {valid.sum():,.0f}  (× {stride*stride:.0f} ≈ full raster)")
            print(f"  % zero:   {(valid == 0).mean() * 100:.1f}")
            print(f"  % > 0:    {(valid > 0).mean() * 100:.1f}")

        # Sample specific locations
        if sample_points:
            from rasterio.warp import transform as rio_transform
            print(f"\nSampled values at specific points:")
            lons = [p[0] for p in sample_points.values()]
            lats = [p[1] for p in sample_points.values()]
            xs, ys = rio_transform("EPSG:4326", src.crs, lons, lats)
            for (label, (lon, lat)), x, y in zip(sample_points.items(), xs, ys):
                try:
                    val = list(src.sample([(x, y)]))[0][0]
                    print(f"  {label:20s} ({lat:.2f}, {lon:.2f}): {val}")
                except Exception as e:
                    print(f"  {label:20s}: error — {e}")

In [ ]:
from rasterio.merge import merge
import rasterio
import gc

srcs = [rasterio.open(f) for f in files]
mosaic, transform = merge(srcs, dtype="float32")

meta = srcs[0].meta.copy()
meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": transform,
    "dtype": "float32",
    "compress": "lzw",
    "tiled": True,
    "BIGTIFF": "IF_NEEDED",
})

with rasterio.open(SOURCE_FILE + r"\pop\ghs_pop_germany.tif", "w", **meta) as dst:
    dst.write(mosaic)

for s in srcs:
    s.close()
del mosaic
gc.collect()

print("Merged raster saved.")

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

# Load Gemeinde shapefile and dissolve to Germany outline
gdf = gpd.read_file(SOURCE_FILE + r"\admin shapefiles\VG5000_GEM.shp", engine="pyogrio")
germany = gdf.dissolve()  # one polygon: all of Germany
# germany = gdf

# Reproject Germany boundary to raster CRS (Mollweide)
with rasterio.open(SOURCE_FILE + r"\pop\ghs_pop_germany.tif") as src:
    germany_mollweide = germany.to_crs(src.crs)
    out_image, out_transform = mask(src, germany_mollweide.geometry, crop=True)
    out_meta = src.meta.copy()

out_meta.update({
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform,
})

with rasterio.open(SOURCE_FILE + r"\pop\ghs_pop_germany_clipped.tif", "w", **out_meta) as dst:
    dst.write(out_image)

print("Clipped to Germany boundary.")

In [ ]:
for f in files:
    inspect_raster(
        f
    )    
    
inspect_raster(
    SOURCE_FILE + r"\pop\ghs_pop_germany_clipped.tif"
)

In [ ]:
ghs_pop_germany_clipped = rasterio.open(SOURCE_FILE + r"\pop\ghs_pop_germany_clipped.tif")

In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd

gdf = gpd.read_file(SOURCE_FILE + r"\admin shapefiles\VG5000_GEM.shp", engine="pyogrio")

gdf_moll = gdf.to_crs("ESRI:54009")

stats = zonal_stats(
    gdf_moll,
    SOURCE_FILE + r"\pop\ghs_pop_germany_clipped.tif",   # note: fewer backslashes
    stats=["sum", "mean", "count"],
    nodata=-200,        # explicit — don't rely on raster metadata
    all_touched=True,   # includes edge cells, safer for small Gemeinden
)

gdf["population"] = [s["sum"] for s in stats]
gdf["mean_density"] = [s["mean"] for s in stats]
gdf["n_cells"] = [s["count"] for s in stats]

# Save as GeoPackage
gdf.to_file(SOURCE_FILE + r"\pop\gemeinden_with_population.gpkg", driver="GPKG", engine="pyogrio")

# Sanity check: top 10 cities
print(gdf[["GEN", "population", "mean_density"]].sort_values("population", ascending=False).head(10))

In [ ]:
print(gdf[["GEN", "population", "mean_density"]].sort_values("population", ascending=False).head(10))

In [ ]:
from scripts.visualizations.plots import plot_gemeinde_population
gdf = gpd.read_file(SOURCE_FILE + r"\admin shapefiles\VG5000_GEM.shp", engine="pyogrio")
plot_gemeinde_population(
    "Berlin", 
    gdf = gdf,
    title="Population distribution in Berlin"
)

In [ ]:
import polars as pl
panel = pl.read_parquet(r"X:\Project-Shares\germart-lite\germart-papaio\panel")

mannheim_stores = panel.filter(
    pl.col("year") == 2024,
    pl.col("Kreis").str.contains("Mannheim"),  # or however you filter
    pl.col("lat").is_not_null(),
).to_pandas()

plot_gemeinde_population(
    "Mannheim",
    gdf=gdf,
    show_stores=mannheim_stores,
    title="Population distribution in Mannheim with store locations"
)
